In [35]:
import pandas as pd
import numpy as np
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [36]:
torch.manual_seed(42)

#### **Loading data**

In [37]:
processed_dir = '../data/processed'
df = pd.read_parquet(f'{processed_dir}/master_data.parquet')
movies = pd.read_parquet(f'{processed_dir}/movies_clean.parquet')

In [38]:
# --- A. GATUNKI (Multi-Hot Encoding) ---
# Rozbijamy 'Action|Comedy' na osobne kolumny zer i jedynek
genres_dummies = movies['genres'].str.get_dummies(sep='|')
# Dodajemy prefix, żeby uniknąć konfliktów nazw
genres_dummies.columns = [f'genre_{c}' for c in genres_dummies.columns]
movies = pd.concat([movies, genres_dummies], axis=1)

# --- B. ROK WYDANIA I DEKADA ---
if 'release_year' not in movies.columns:
    movies['release_year'] = movies['title'].str.extract(r'\((\d{4})\)').astype(float)

movies['release_year'] = movies['release_year'].fillna(movies['release_year'].median())
movies['decade'] = (movies['release_year'] // 10 * 10).astype(int)

# --- C. STATYSTYKI HISTORYCZNE FILMU ---
# Jak popularny i jak dobry jest ten film obiektywnie?
movie_stats = df.groupby('movieId').agg(
    movie_rating_count=('rating', 'count'),
    movie_rating_mean=('rating', 'mean'),
    movie_rating_std=('rating', 'std') # Mierzy polaryzację filmu (kontrowersyjność)
).reset_index()

movie_stats['movie_rating_std'] = movie_stats['movie_rating_std'].fillna(0)
movies = movies.merge(movie_stats, on='movieId', how='left')

In [39]:
# --- D. STATYSTYKI BEHAWIORALNE UŻYTKOWNIKA ---
user_stats = df.groupby('userId').agg(
    user_rating_count=('rating', 'count'), # Aktywność użytkownika
    user_rating_mean=('rating', 'mean'),   # Średnia wystawiana ocena (User Bias)
    user_rating_std=('rating', 'std')      # Czy używa całej skali, czy tylko 4 i 5?
).reset_index()

user_stats['user_rating_std'] = user_stats['user_rating_std'].fillna(0)

# Jedyne złączenie statystyk użytkownika
df = df.merge(user_stats, on='userId', how='left')

# --- E. ŁĄCZENIE DANYCH FILMU DO GŁÓWNEGO DF ---
# UWAGA: Usunięto 'release_year' z cols_to_merge, bo df z pliku master_data już to ma!
cols_to_merge = ['movieId', 'decade', 'movie_rating_count', 'movie_rating_mean', 'movie_rating_std'] + list(genres_dummies.columns)
df = df.merge(movies[cols_to_merge], on='movieId', how='left')

# --- F. CECHY KONTEKSTOWE ---
# Wiek filmu w momencie wystawiania oceny
df['rating_year'] = pd.to_datetime(df['datetime']).dt.year
# Używamy prostej funkcji clip() do usunięcia wartości ujemnych
df['movie_age_at_rating'] = (df['rating_year'] - df['release_year']).clip(lower=0)

# --- G. NORMALIZACJA CECH CIĄGŁYCH ---
continuous_features = [
    'user_rating_count', 'user_rating_mean', 'user_rating_std',
    'movie_rating_count', 'movie_rating_mean', 'movie_rating_std',
    'release_year', 'movie_age_at_rating'
]

df[continuous_features] = df[continuous_features].fillna(0)
df.replace([np.inf, -np.inf], 0, inplace=True)

scaler = StandardScaler()
df[continuous_features] = scaler.fit_transform(df[continuous_features])

# --- H. KODOWANIE ID (Label Encoding) ---
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()
decade_encoder = LabelEncoder()

df['user_idx'] = user_encoder.fit_transform(df['userId'])
df['item_idx'] = item_encoder.fit_transform(df['movieId'])
df['decade_idx'] = decade_encoder.fit_transform(df['decade'])

In [40]:
num_users = df['user_idx'].nunique()
num_items = df['item_idx'].nunique()
num_decades = df['decade_idx'].nunique()

print(f"Użytkowników: {num_users}, Filmów: {num_items}, Dekad: {num_decades}")

Użytkowników: 610, Filmów: 9724, Dekad: 12


In [41]:
# --- PRZYGOTOWANIE LIST KOLUMN (aby łatwo przekazać je do Datasetu) ---
genre_cols = [c for c in df.columns if c.startswith('genre_')]

# Zakładam, że lista 'continuous_features' z poprzedniego kroku jest nadal w pamięci:
# continuous_features = ['user_rating_count', 'user_rating_mean', ...]

# 4. Podział chronologiczny
df = df.sort_values('datetime')
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

# 5. Aktualizacja Datasetu (Dodanie obsługi wektorów ciągłych i Multi-Hot)
class AdvancedHybridMovieDataset(Dataset):
    def __init__(self, users, items, decades, continuous_feats, genres_multi, ratings):
        # Cechy kategoryczne (ID) - typ long dla warstw nn.Embedding
        self.users = torch.tensor(users, dtype=torch.long)
        self.items = torch.tensor(items, dtype=torch.long)
        self.decades = torch.tensor(decades, dtype=torch.long)
        
        # Cechy gęste/ciągłe i wektory 0/1 gatunków - typ float32 dla warstw nn.Linear
        self.continuous_feats = torch.tensor(continuous_feats, dtype=torch.float32)
        self.genres_multi = torch.tensor(genres_multi, dtype=torch.float32) 
        
        # Cel (Target)
        self.ratings = torch.tensor(ratings, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return (
            self.users[idx], 
            self.items[idx], 
            self.decades[idx], 
            self.continuous_feats[idx], 
            self.genres_multi[idx], 
            self.ratings[idx]
        )

# 6. Aktualizacja Loaderów
train_loader = DataLoader(
    AdvancedHybridMovieDataset(
        train_df['user_idx'].values, 
        train_df['item_idx'].values, 
        train_df['decade_idx'].values,
        train_df[continuous_features].values, # Przekazujemy macierz cech ciągłych
        train_df[genre_cols].values,          # Przekazujemy macierz gatunków Multi-Hot
        train_df['rating'].values
    ), 
    batch_size=1024, 
    shuffle=True,
    drop_last=True
)

# Podobnie test_loader (bądź val_loader dla Early Stoppingu)
test_loader = DataLoader(
    AdvancedHybridMovieDataset(
        test_df['user_idx'].values, 
        test_df['item_idx'].values, 
        test_df['decade_idx'].values,
        test_df[continuous_features].values, 
        test_df[genre_cols].values, 
        test_df['rating'].values
    ), 
    batch_size=1024, 
    shuffle=False,
    drop_last=True
)

In [42]:
class AdvancedHybridTwoTowerModel(nn.Module):
    def __init__(self, num_users, num_items, num_decades, num_continuous_feats, num_genre_cols, embed_dim=64, dropout_rate=0.2, global_mean=3.5):
        super().__init__()
        
        self.global_mean = global_mean
        
        # --- 1. WIEŻA UŻYTKOWNIKA ---
        self.user_embed = nn.Embedding(num_users, embed_dim)
        # Zakładamy, że z continuous_feats pierwsze 3 to cechy użytkownika (count, mean, std)
        self.user_dense_process = nn.Sequential(
            nn.Linear(3, 16),
            nn.GELU()
        )
        self.user_tower = nn.Sequential(
            nn.Linear(embed_dim + 16, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64)
        )
        
        # --- 2. WIEŻA FILMU (HYBRYDOWA) ---
        self.item_embed = nn.Embedding(num_items, embed_dim)
        self.decade_embed = nn.Embedding(num_decades, embed_dim)
        
        # Przetwarzanie wektorów Multi-Hot gatunków (zamiast Embedding)
        self.genre_process = nn.Sequential(
            nn.Linear(num_genre_cols, embed_dim),
            nn.GELU()
        )
        
        # Przetwarzanie pozostałych 5 cech ciągłych (o filmie i kontekście)
        self.item_dense_process = nn.Sequential(
            nn.Linear(5, 32),
            nn.GELU()
        )
        
        self.item_tower = nn.Sequential(
            nn.Linear(embed_dim * 3 + 32, 256), # ID + Dekada + Gatunki + Cechy Gęste
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64)
        )
        
        # --- 3. BIASY (KRYTYCZNE) ---
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        
        # --- 4. WARSTWA PREDYKCYJNA ---
        self.prediction_layer = nn.Sequential(
            nn.Linear(64 + 64, 64),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 32),
            nn.GELU(),
            nn.Linear(32, 1)
        )
        
        # Inicjalizacja (tylko Embeddingi utrzymujemy małe, w biasach startujemy od 0)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_idx, item_idx, decade_idx, continuous_feats, genres_multi):
        # 1. Przetwarzanie cech użytkownika
        u_emb = self.user_embed(user_idx)
        u_dense = self.user_dense_process(continuous_feats[:, :3]) # Wyciągamy pierwsze 3 cechy
        u_combined = torch.cat([u_emb, u_dense], dim=1)
        u_vector = self.user_tower(u_combined)
        
        # 2. Przetwarzanie cech filmu
        i_emb = self.item_embed(item_idx)
        d_emb = self.decade_embed(decade_idx)
        g_emb = self.genre_process(genres_multi)
        i_dense = self.item_dense_process(continuous_feats[:, 3:]) # Wyciągamy pozostałe 5 cech
        
        i_combined = torch.cat([i_emb, d_emb, g_emb, i_dense], dim=1)
        i_vector = self.item_tower(i_combined)
        
        # 3. Fuzja i predykcja
        combined = torch.cat([u_vector, i_vector], dim=1)
        out = self.prediction_layer(combined)
        
        # 4. Dodanie biasów i średniej globalnej
        u_b = self.user_bias(user_idx)
        i_b = self.item_bias(item_idx)
        
        # Wynik to średnia globalna + odchylenie użytkownika + odchylenie filmu + interakcja z wież
        final_out = self.global_mean + u_b + i_b + out
        return final_out.squeeze()

In [43]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Używamy modelu przyjmującego nowe cechy ciągłe i multi-hot
model = AdvancedHybridTwoTowerModel(
    num_users=num_users,
    num_items=num_items,
    num_decades=num_decades,
    num_continuous_feats=len(continuous_features), 
    num_genre_cols=len(genre_cols),              
    embed_dim=64, 
    global_mean=train_df['rating'].mean()
).to(device)

# 🔥 LEPSZY LOSS
criterion = nn.SmoothL1Loss()

# 🔥 LEPSZY OPTIMIZER
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,              
    weight_decay=1e-4     
)

# 🔥 SCHEDULER
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=1
)

epochs = 8

# 🔥 ZAPIS NAJLEPSZEGO MODELU (Zamiast ucinania pętli)
best_val_loss = float('inf')
best_model_state = None

print(f"Rozpoczynanie treningu rozszerzonego modelu na {device}...")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    start_time = time.time()
    
    # 🔴 POPRAWKA: Prawidłowe rozpakowanie 6 elementów z loadera!
    for users, items, decades, continuous_feats, genres_multi, ratings in train_loader:
        users = users.to(device)
        items = items.to(device)
        decades = decades.to(device)
        continuous_feats = continuous_feats.to(device)
        genres_multi = genres_multi.to(device)
        ratings = ratings.to(device)
        
        optimizer.zero_grad()
        
        # Przekazanie nowych argumentów
        predictions = model(users, items, decades, continuous_feats, genres_multi)
        loss = criterion(predictions, ratings)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        total_loss += loss.item()
        
    avg_train_loss = total_loss / len(train_loader)

    # 🔴 WALIDACJA 
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for users, items, decades, continuous_feats, genres_multi, ratings in test_loader:  
            users = users.to(device)
            items = items.to(device)
            decades = decades.to(device)
            continuous_feats = continuous_feats.to(device)
            genres_multi = genres_multi.to(device)
            ratings = ratings.to(device)

            preds = model(users, items, decades, continuous_feats, genres_multi)
            loss = criterion(preds, ratings)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(test_loader)

    # 🔥 scheduler patrzy na WALIDACJĘ
    scheduler.step(avg_val_loss)

    epoch_time = time.time() - start_time

    print(
        f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train: {avg_train_loss:.4f} | "
        f"Val: {avg_val_loss:.4f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.6f} | "
        f"Czas: {epoch_time:.2f}s"
    )

    # 🔥 Zapisujemy wagi najlepszego modelu
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        # używamy .copy() dla pewności, że wagi zostaną sklonowane w pamięci
        best_model_state = model.state_dict().copy() 

# 🔥 Po 8 epokach przywróć najlepszy model
if best_model_state is not None:
    model.load_state_dict(best_model_state)


# =====================
# 📊 TEST NA ZBIORZE TESTOWYM
# =====================
model.eval()
test_predictions, test_actuals = [], []

with torch.no_grad():
    for users, items, decades, continuous_feats, genres_multi, ratings in test_loader:
        users = users.to(device)
        items = items.to(device)
        decades = decades.to(device)
        continuous_feats = continuous_feats.to(device)
        genres_multi = genres_multi.to(device)

        preds = model(users, items, decades, continuous_feats, genres_multi)

        # 🔥 CLAMP
        preds = torch.clamp(preds, 0.5, 5.0)

        test_predictions.extend(preds.cpu().numpy())
        
        # Pamiętaj, aby przenieść oceny z GPU/Tenzora z powrotem na CPU przed numpy()
        test_actuals.extend(ratings.cpu().numpy())

rmse = np.sqrt(mean_squared_error(test_actuals, test_predictions))
print(f"\nEnhanced Hybrid Two-Tower RMSE: {rmse:.4f}")

Rozpoczynanie treningu rozszerzonego modelu na cpu...
Epoch 01/8 | Train: 0.3214 | Val: 0.2712 | LR: 0.000300 | Czas: 11.77s
Epoch 02/8 | Train: 0.2811 | Val: 0.2683 | LR: 0.000300 | Czas: 7.94s
Epoch 03/8 | Train: 0.2751 | Val: 0.2690 | LR: 0.000300 | Czas: 7.59s
Epoch 04/8 | Train: 0.2716 | Val: 0.2711 | LR: 0.000150 | Czas: 8.16s
Epoch 05/8 | Train: 0.2664 | Val: 0.2745 | LR: 0.000150 | Czas: 7.46s
Epoch 06/8 | Train: 0.2629 | Val: 0.2786 | LR: 0.000075 | Czas: 8.03s
Epoch 07/8 | Train: 0.2603 | Val: 0.2831 | LR: 0.000075 | Czas: 8.80s
Epoch 08/8 | Train: 0.2579 | Val: 0.2858 | LR: 0.000037 | Czas: 8.58s

Enhanced Hybrid Two-Tower RMSE: 0.8298
